System information (for reproducibility):

In [18]:
versioninfo()

Julia Version 1.12.6
Commit 15346901f00 (2026-04-09 19:20 UTC)
Build Info:
  Official https://julialang.org release
Platform Info:
  OS: macOS (arm64-apple-darwin24.0.0)
  CPU: 12 × Apple M2 Max
  WORD_SIZE: 64
  LLVM: libLLVM-18.1.7 (ORCJIT, apple-m2)
  GC: Built with stock GC
Threads: 8 default, 1 interactive, 8 GC (on 8 virtual cores)
Environment:
  JULIA_NUM_THREADS = 8
  JULIA_EDITOR = code


Load packages:

In [19]:
using Pkg

Pkg.activate(pwd())
Pkg.instantiate()
Pkg.status()

  Activating project at `~/Documents/github.com/ucla-biostat-257/2026spring/slides/18-cg`


Status `~/Documents/github.com/ucla-biostat-257/2026spring/slides/18-cg/Project.toml`
  [2169fc97] AlgebraicMultigrid v1.2.0
  [7d9fca2a] Arpack v0.5.4
  [6e4b80f9] BenchmarkTools v1.8.0
  [42fd0dbc] IterativeSolvers v0.9.4
  [ba0b0d4f] Krylov v0.10.6
  [7a12625a] LinearMaps v3.11.4
  [b51810bb] MatrixDepot v1.0.15
  [af69fa37] Preconditioners v0.6.2
  [b8865327] UnicodePlots v3.8.2
  [37e2e46d] LinearAlgebra v1.12.0
  [9a3f8284] Random v1.11.0
  [2f01184e] SparseArrays v1.12.0


In [20]:
using AlgebraicMultigrid, BenchmarkTools, IterativeSolvers, 
    Krylov, LinearAlgebra, MatrixDepot, Random, SparseArrays

## Introduction

* Conjugate gradient is the top-notch iterative method for solving large, **structured** linear systems $\mathbf{A} \mathbf{x} = \mathbf{b}$, where $\mathbf{A}$ is pd.  
Earlier we talked about Jacobi, Gauss-Seidel, and successive over-relaxation (SOR) as the classical iterative solvers. They are rarely used in practice due to slow convergence.  

    [Kershaw's results](http://www.sciencedirect.com/science/article/pii/0021999178900980?via%3Dihub) for a fusion problem.

| Method                                 | Number of Iterations |
|----------------------------------------|----------------------|
| Gauss Seidel                           | 208,000              |
| Block SOR methods                      | 765                  |
| Incomplete Cholesky **conjugate gradient** | 25                   |


* History: Hestenes (**UCLA** professor!) and Stiefel proposed conjugate gradient method in 1950s.

Hestenes and Stiefel (1952), [Methods of conjugate gradients for solving linear systems](http://nvlpubs.nist.gov/nistpubs/jres/049/jresv49n6p409_A1b.pdf), _Jounral of Research of the National Bureau of Standards_.

* Solve linear equation $\mathbf{A} \mathbf{x} = \mathbf{b}$, where $\mathbf{A} \in \mathbb{R}^{n \times n}$ is **pd**, is equivalent to 
$$
\begin{eqnarray*}
	\text{minimize} \,\, f(\mathbf{x}) = \frac 12 \mathbf{x}^T \mathbf{A} \mathbf{x} - \mathbf{b}^T \mathbf{x}.
\end{eqnarray*}
$$
Denote $\nabla f(\mathbf{x}) = \mathbf{A} \mathbf{x} - \mathbf{b} =: r(\mathbf{x})$.

## Conjugate gradient (CG) method

* Consider a simple idea: coordinate descent, that is to update components $x_j$ alternatingly. Same as the Gauss-Seidel iteration. Usually it takes too many iterations.

<img src="coordinate_descent.png" width="400" align="center"/>

* A set of vectors $\{\mathbf{p}^{(0)},\ldots,\mathbf{p}^{(t)}\}$ are said to be **conjugate with respect to $\mathbf{A}$** if
$$
\begin{eqnarray*}
	\mathbf{p}_i^T \mathbf{A} \mathbf{p}_j = 0, \quad \text{for all } i \ne j.
\end{eqnarray*}
$$
For example, eigen-vectors of $\mathbf{A}$ are conjugate to each other. Why?

* **Conjugate direction** method: Given a set of conjugate vectors $\{\mathbf{p}^{(0)},\ldots,\mathbf{p}^{(n-1)}\}$, at iteration $t$, we search along the conjugate direction $\mathbf{p}^{(t)}$
$$
\begin{eqnarray*}
	\mathbf{x}^{(t+1)} = \mathbf{x}^{(t)} + \alpha^{(t)} \mathbf{p}^{(t)},
\end{eqnarray*}
$$
where
$$
\begin{eqnarray*}
	\alpha^{(t)} = - \frac{\mathbf{r}^{(t)T} \mathbf{p}^{(t)}}{\mathbf{p}^{(t)T} \mathbf{A} \mathbf{p}^{(t)}}
\end{eqnarray*}
$$
is the optimal step length.

* Theorem: In conjugate direction method, $\mathbf{x}^{(t)}$ converges to the solution in **at most** $n$ steps.

    Intuition: Look at graph.
    
<img src="conjugate_direction.png" width="400" align="center"/>

* **Conjugate gradient** method. Idea: generate $\mathbf{p}^{(t)}$ using only $\mathbf{p}^{(t-1)}$
$$
\begin{eqnarray*}
	\mathbf{p}^{(t)} = - \mathbf{r}^{(t)} + \beta^{(t)} \mathbf{p}^{(t-1)},
\end{eqnarray*}
$$
where $\beta^{(t)}$ is determined by the conjugacy condition $\mathbf{p}^{(t-1)T} \mathbf{A} \mathbf{p}^{(t)} = 0$
$$
\begin{eqnarray*}
	\beta^{(t)} = \frac{\mathbf{r}^{(t)T} \mathbf{A} \mathbf{p}^{(t-1)}}{\mathbf{p}^{(t-1)T} \mathbf{A} \mathbf{p}^{(t-1)}}.
\end{eqnarray*}
$$

* **CG algorithm (preliminary version)**:  

    0. Given $\mathbf{x}^{(0)}$
    0. Initialize: $\mathbf{r}^{(0)} \gets \mathbf{A} \mathbf{x}^{(0)} - \mathbf{b}$, $\mathbf{p}^{(0)} \gets - \mathbf{r}^{(0)}$, $t=0$
    0. While $\mathbf{r}^{(t)} \ne \mathbf{0}$
        1. $\alpha^{(t)} \gets - \frac{\mathbf{r}^{(t)T} \mathbf{p}^{(t)}}{\mathbf{p}^{(t)T} \mathbf{A} \mathbf{p}^{(t)}}$
        2. $\mathbf{x}^{(t+1)} \gets \mathbf{x}^{(t)} + \alpha^{(t)} \mathbf{p}^{(t)}$
        3. $\mathbf{r}^{(t+1)} \gets \mathbf{A} \mathbf{x}^{(t+1)} - \mathbf{b}$
        4. $\beta^{(t+1)} \gets \frac{\mathbf{r}^{(t+1)T} \mathbf{A} \mathbf{p}^{(t)}}{\mathbf{p}^{(t)T} \mathbf{A} \mathbf{p}^{(t)}}$
        5. $\mathbf{p}^{(t+1)} \gets - \mathbf{r}^{(t+1)} + \beta^{(t+1)} \mathbf{p}^{(t)}$
        6. $t \gets t+1$
        
    Remark: The initial conjugate direction $\mathbf{p}^{(0)} \gets - \mathbf{r}^{(0)}$ is crucial.
        
* Theorem: With CG algorithm
    0. $\mathbf{r}^{(t)}$ are mutually orthogonal. 
    0. $\{\mathbf{r}^{(0)},\ldots,\mathbf{r}^{(t)}\}$ is contained in the **Krylov subspace** of degree $t$ for $\mathbf{r}^{(0)}$, denoted by
    $$
    \begin{eqnarray*}
        {\cal K}(\mathbf{r}^{(0)}; t) = \text{span} \{\mathbf{r}^{(0)},\mathbf{A} \mathbf{r}^{(0)}, \mathbf{A}^2 \mathbf{r}^{(0)}, \ldots, \mathbf{A}^{t} \mathbf{r}^{(0)}\}.
    \end{eqnarray*}
    $$
    0. $\{\mathbf{p}^{(0)},\ldots,\mathbf{p}^{(t)}\}$ is contained in ${\cal K}(\mathbf{r}^{(0)}; t)$. 
    0. $\mathbf{p}^{(0)}, \ldots, \mathbf{p}^{(t)}$ are conjugate with respect to $\mathbf{A}$.  
The iterates $\mathbf{x}^{(t)}$ converge to the solution in at most $n$ steps.

* **CG algorithm (economical version)**: saves one matrix-vector multiplication.

    0. Given $\mathbf{x}^{(0)}$
    0. Initialize: $\mathbf{r}^{(0)} \gets \mathbf{A} \mathbf{x}^{(0)} - \mathbf{b}$, $\mathbf{p}^{(0)} \gets - \mathbf{r}^{(0)}$, $t=0$
    0. While $\mathbf{r}^{(t)} \ne \mathbf{0}$
        1. $\alpha^{(t)} \gets \frac{\mathbf{r}^{(t)T} \mathbf{r}^{(t)}}{\mathbf{p}^{(t)T} \mathbf{A} \mathbf{p}^{(t)}}$
        2. $\mathbf{x}^{(t+1)} \gets \mathbf{x}^{(t)} + \alpha^{(t)} \mathbf{p}^{(t)}$
        3. $\mathbf{r}^{(t+1)} \gets \mathbf{r}^{(t)} + \alpha^{(t)} \mathbf{A} \mathbf{p}^{(t)}$
        4. $\beta^{(t+1)} \gets \frac{\mathbf{r}^{(t+1)T} \mathbf{r}^{(t+1)}}{\mathbf{r}^{(t)T} \mathbf{r}^{(t)}}$
        5. $\mathbf{p}^{(t+1)} \gets - \mathbf{r}^{(t+1)} + \beta^{(t+1)} \mathbf{p}^{(t)}$
        6. $t \gets t+1$

* Computation cost per iteration is **one** matrix vector multiplication: $\mathbf{A} \mathbf{p}^{(t)}$.  
Consider PageRank problem, $\mathbf{A}$ has dimension $n \approx 10^{10}$ but is highly structured (sparse + low rank). Each matrix vector multiplication takes $O(n)$.
    
* Theorem: If $\mathbf{A}$ has $r$ distinct eigenvalues, $\mathbf{x}^{(t)}$ converges to solution $\mathbf{x}^*$ in at most $r$ steps.

## Pre-conditioned conjugate gradient (PCG)

* Summary of conjugate gradient method for solving $\mathbf{A} \mathbf{x} = \mathbf{b}$ or equivalently minimizing $\frac 12 \mathbf{x}^T \mathbf{A} \mathbf{x} -  \mathbf{b}^T \mathbf{x}$:
    * Each iteration needs one matrix vector multiplication: $\mathbf{A} \mathbf{p}^{(t+1)}$. For structured $\mathbf{A}$, often $O(n)$ cost per iteration.
    * Guaranteed to converge in $n$ steps.
    
* Two important bounds for conjugate gradient algorithm:

    Let $\lambda_1 \le \cdots \le \lambda_n$ be the ordered eigenvalues of a pd $\mathbf{A}$.  
$$
\begin{eqnarray*}
    \|\mathbf{x}^{(t+1)} - \mathbf{x}^*\|_{\mathbf{A}}^2 &\le& \left( \frac{\lambda_{n-t} - \lambda_1}{\lambda_{n-t} + \lambda_1} \right)^2 \|\mathbf{x}^{(0)} - \mathbf{x}^*\|_{\mathbf{A}}^2 \\
    \|\mathbf{x}^{(t+1)} - \mathbf{x}^*\|_{\mathbf{A}}^2 &\le& 2 \left( \frac{\sqrt{\kappa(\mathbf{A})}-1}{\sqrt{\kappa(\mathbf{A})}+1} \right)^{t} \|\mathbf{x}^{(0)} - \mathbf{x}^*\|_{\mathbf{A}}^2,
\end{eqnarray*}
$$
where $\kappa(\mathbf{A}) = \lambda_n/\lambda_1$ is the condition number of $\mathbf{A}$.

<img src="cg_twocluster_spectrum.png" width="300" align="center"/>

<img src="cg_twocluster_iterates.png" width="300" align="center"/>

* Messages:
    * Roughly speaking, if the eigenvalues of $\mathbf{A}$ occur in $r$ distinct clusters, the CG iterates will _approximately_ solve the problem after $O(r)$ steps.  
    * $\mathbf{A}$ with a small condition number ($\lambda_1 \approx \lambda_n$) converges fast.
    
* **Pre-conditioning**: Change of variables $\widehat{\mathbf{x}} = \mathbf{C} \mathbf{x}$ via a nonsingular $\mathbf{C}$ and solve
$$
	(\mathbf{C}^{-T} \mathbf{A} \mathbf{C}^{-1}) \widehat{\mathbf{x}} = \mathbf{C}^{-T} \mathbf{b}.
$$
Choose $\mathbf{C}$ such that 
    * $\mathbf{C}^{-T} \mathbf{A} \mathbf{C}^{-1}$ has small condition number, or 
    * $\mathbf{C}^{-T} \mathbf{A} \mathbf{C}^{-1}$ has clustered eigenvalues
    * Inexpensive solution of $\mathbf{C}^T \mathbf{C} \mathbf{y} = \mathbf{r}$
    
* Preconditioned CG does not make use of $\mathbf{C}$ explicitly, but rather the matrix $\mathbf{M} = \mathbf{C}^T \mathbf{C}$.

* **Preconditioned CG (PCG)** algorithm: 

    0. Given $\mathbf{x}^{(0)}$, pre-conditioner $\mathbf{M}$
    0. $\mathbf{r}^{(0)} \gets \mathbf{A} \mathbf{x}^{(0)} - \mathbf{b}$
    0. solve $\mathbf{M} \mathbf{y}^{(0)} = \mathbf{r}^{(0)}$ for $\mathbf{y}^{(0)}$
    0. $\mathbf{p}^{(0)} \gets - \mathbf{r}^{(0)}$, $t=0$
    0. While $\mathbf{r}^{(t)} \ne \mathbf{0}$
        1. $\alpha^{(t)} \gets \frac{\mathbf{r}^{(t)T} \mathbf{y}^{(t)}}{\mathbf{p}^{(t)T} \mathbf{A} \mathbf{p}^{(t)}}$
        2. $\mathbf{x}^{(t+1)} \gets \mathbf{x}^{(t)} + \alpha^{(t)} \mathbf{p}^{(t)}$
        3. $\mathbf{r}^{(t+1)} \gets \mathbf{r}^{(t)} + \alpha^{(t)} \mathbf{A} \mathbf{p}^{(t)}$
        4. Solve $\mathbf{M} \mathbf{y}^{(t+1)} = \mathbf{r}^{(t+1)}$ for $\mathbf{y}^{(t+1)}$
        5. $\beta^{(t+1)} \gets \frac{\mathbf{r}^{(t+1)T} \mathbf{y}^{(t+1)}}{\mathbf{r}^{(t)T} \mathbf{r}^{(t)}}$
        6. $\mathbf{p}^{(t+1)} \gets - \mathbf{y}^{(t+1)} + \beta^{(t+1)} \mathbf{p}^{(t)}$
        7. $t \gets t+1$

    Remark: Only extra cost in the pre-conditioned CG algorithm is the need to solve the linear system $\mathbf{M} \mathbf{y} = \mathbf{r}$.
    
* Pre-conditioning is more like an art than science. Some choices include     
    * Incomplete Cholesky. $\mathbf{A} \approx \tilde{\mathbf{L}} \tilde{\mathbf{L}}^T$, where $\tilde{\mathbf{L}}$ is a sparse approximate Cholesky factor. Then $\tilde{\mathbf{L}}^{-1} \mathbf{A} \tilde{\mathbf{L}}^{-T} \approx \mathbf{I}$ (perfectly conditioned) and $\mathbf{M} \mathbf{y} = \tilde{\mathbf{L}} \tilde {\mathbf{L}}^T \mathbf{y} = \mathbf{r}$ is easy to solve.  
    * Banded pre-conditioners.  
    * Choose $\mathbf{M}$ as a coarsened version of $\mathbf{A}$.
    * Subject knowledge. Knowledge about the structure and origin of a problem is often the key to devising efficient pre-conditioner. For example, see recent work of Stein, Chen, Anitescu (2012) for pre-conditioning large covariance matrices. http://epubs.siam.org/doi/abs/10.1137/110834469

### Example of PCG

[Preconditioners.jl](https://github.com/mohamed82008/Preconditioners.jl) wraps a bunch of preconditioners.

We use the Wathen matrix (sparse and positive definite) as a test matrix.

In [21]:
# Wathen matrix of dimension 30401 x 30401
A = matrixdepot("wathen", 100)

30401×30401 SparseMatrixCSC{Float64, Int64} with 471601 stored entries:
⎡⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⎦

In [22]:
using UnicodePlots
spy(A)

          ┌──────────────────────────────┐    
        1 │⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ > 0
          │⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ < 0
          │⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀│    
          │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀│    
   30 401 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦│    
          └──────────────────────────────┘    
          ⠀1⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀30 401⠀    
          ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀471 601 ≠ 0⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀    

In [23]:
# sparsity level
count(!iszero, A) / length(A)

0.0005102687577359558

In [24]:
# rhs
b = ones(size(A, 1))
# solve Ax=b by CG
xcg = IterativeSolvers.cg(A, b);
@benchmark IterativeSolvers.cg($A, $b)

BenchmarkTools.Trial: 38 samples with 1 evaluation per sample.
 Range (min … max):  128.222 ms … 156.752 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     130.010 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   133.061 ms ±   6.697 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄▂█                                                            
  █████▆▁▁▁▆▁▄▄▄▄▄▁▁▄▆▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▄▁▁▁▁▄▁▁▄▁▁▁▁▁▁▁▁▁▁▁▁▁▄ ▁
  128 ms           Histogram: frequency by time          157 ms <

 Memory estimate: 960.89 KiB, allocs estimate: 20.

#### Diagonal preconditioner

Compute the diagonal preconditioner:

In [25]:
using Preconditioners

# Diagonal preconditioner
@time p = DiagonalPreconditioner(A)
dump(p)

  0.000249 seconds (47 allocations: 1.748 MiB)
DiagonalPreconditioner{Float64, Vector{Float64}}
  D: Array{Float64}((30401,)) [10.631169678824207, 56.69957162039578, 11.290439169633146, 3.5161039509810066, 6.232098071560602, 29.721752430675537, 16.15884882445218, 56.458774633069424, 11.466098592333358, 4.693751192708488  …  54.41939805039397, 18.749689367168244, 45.578945241170004, 15.94204525051839, 39.445296094928075, 16.736778439089665, 49.81752224688348, 12.311043667230436, 15.841377311678846, 2.9702582459397835]


In [26]:
# solver Ax=b by PCG
xpcg = IterativeSolvers.cg(A, b, Pl = p)
# same answer?
norm(xcg - xpcg)

7.897206418903282e-7

In [27]:
# PCG with diagonal preconditioner is >5 fold faster than CG
@benchmark IterativeSolvers.cg($A, $b, Pl = $p)

BenchmarkTools.Trial: 320 samples with 1 evaluation per sample.
 Range (min … max):  14.124 ms … 20.363 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     15.422 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   15.669 ms ±  1.162 ms  ┊ GC (mean ± σ):  0.31% ± 1.57%

     ▁        ▁  ▃█                                            
  ▇▆▆█▄▃▂▄▄▄▆▇█▇▄██▅▆▄▇▄▃▄▄▂▃▂▃▄▄▃▃▃▃▃▄▄▃▃▂▃▁▃▃▃▂▄▄▂▄▃▁▂▁▁▃▁▂ ▃
  14.1 ms         Histogram: frequency by time        18.9 ms <

 Memory estimate: 960.89 KiB, allocs estimate: 20.

#### Incomplete Cholesky preconditioner

Compute the incomplete cholesky preconditioner:

In [28]:
@time p = CholeskyPreconditioner(A, 2)
dump(p)

  0.019022 seconds (90 allocations: 24.573 MiB, 3.37% gc time)
CholeskyPreconditioner{LimitedLDLFactorizations.LimitedLDLFactorization{Float64, Int64, Vector{Int64}, Vector{Int64}}}
  ldlt: LimitedLDLFactorizations.LimitedLDLFactorization{Float64, Int64, Vector{Int64}, Vector{Int64}}
    __factorized: Bool true
    n: Int64 30401
    colptr: Array{Int64}((30402,)) [1, 13, 25, 37, 54, 65, 77, 89, 101, 118  …  265007, 265010, 265014, 265017, 265021, 265024, 265026, 265027, 265027, 265027]
    rowind: Array{Int64}((281402,)) [3, 4, 5, 11, 12, 1557, 1558, 1559, 1599, 4639  …  30394, 30395, 30396, 30396, 30397, 30398, 30398, 30399, 30400, 30400]
    Lrowind: SubArray{Int64, 1, Vector{Int64}, Tuple{UnitRange{Int64}}, true}
      parent: Array{Int64}((281402,)) [3, 4, 5, 11, 12, 1557, 1558, 1559, 1599, 4639  …  30394, 30395, 30396, 30396, 30397, 30398, 30398, 30399, 30400, 30400]
      indices: Tuple{UnitRange{Int64}}
        1: UnitRange{Int64}
          start: Int64 1
          stop: Int64 

Pre-conditioned conjugate gradient:

In [29]:
# solver Ax=b by PCG
xpcg = IterativeSolvers.cg(A, b, Pl = p)
# same answer?
norm(xcg - xpcg)

7.904831876056511e-7

In [30]:
# PCG with incomplete Cholesky is >5 fold faster than CG
@benchmark IterativeSolvers.cg($A, $b, Pl = $p)

BenchmarkTools.Trial: 307 samples with 1 evaluation per sample.
 Range (min … max):  15.710 ms …  22.001 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     16.118 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   16.357 ms ± 612.693 μs  ┊ GC (mean ± σ):  0.16% ± 0.70%

      ▂▇▂█▄▄                                                    
  ▅▃▆▅████████▆▅▄▅▆▄▂▄▄▆▆▂▃▃▃▃▁▄▄▄▂▃▃▃▃▃▃▃▄▃▄▂▂▁▃▁▁▁▃▁▁▁▁▁▁▁▁▂ ▃
  15.7 ms         Histogram: frequency by time         18.2 ms <

 Memory estimate: 960.89 KiB, allocs estimate: 20.

#### AMG preconditioner

Let's try the AMG preconditioner.

In [31]:
using AlgebraicMultigrid

@time ml = AMGPreconditioner{RugeStuben}(A) # Construct a Ruge-Stuben solver

  0.024700 seconds (923 allocations: 67.430 MiB, 7.47% gc time)


AMGPreconditioner{RugeStuben, AlgebraicMultigrid.MultiLevel{AlgebraicMultigrid.Pinv{Float64}, GaussSeidel{SymmetricSweep}, GaussSeidel{SymmetricSweep}, SparseMatrixCSC{Float64, Int64}, Adjoint{Float64, SparseMatrixCSC{Float64, Int64}}, SparseMatrixCSC{Float64, Int64}, AlgebraicMultigrid.MultiLevelWorkspace{Vector{Float64}, 1}}, AlgebraicMultigrid.V}(Multilevel Solver
-----------------
Operator Complexity: 1.135
Grid Complexity: 1.172
No. of Levels: 8
Coarse Solver: Pinv
Level     Unknowns     NonZeros
-----     --------     --------
    1        30401       471601 [88.12%]
    2         3609        45321 [ 8.47%]
    3         1086        12918 [ 2.41%]
    4          352         3726 [ 0.70%]
    5          125         1181 [ 0.22%]
    6           42          318 [ 0.06%]
    7           16          118 [ 0.02%]
    8            6           26 [ 0.00%]
, AlgebraicMultigrid.V())

In [32]:
# use AMG preconditioner in CG
xamg = IterativeSolvers.cg(A, b, Pl = ml)
# same answer?
norm(xcg - xamg)

7.88198777410114e-7

In [33]:
@benchmark IterativeSolvers.cg($A, $b, Pl = $ml)

BenchmarkTools.Trial: 98 samples with 1 evaluation per sample.
 Range (min … max):  50.139 ms …  54.660 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     51.128 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   51.304 ms ± 764.610 μs  ┊ GC (mean ± σ):  0.31% ± 0.50%

        ▂  ▁ ▁█▂▂       ▂                                       
  ▃▃▃▅▆██▃▅█▆████▆██▆▅▃▁█▃▆▁▅▅▃▃▁▃▁▅▁▆▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▁
  50.1 ms         Histogram: frequency by time         54.4 ms <

 Memory estimate: 4.87 MiB, allocs estimate: 468.

## Other Krylov subspace methods

* We leant about CG/PCG, which is for solving $\mathbf{A} \mathbf{x} = \mathbf{b}$, $\mathbf{A}$ pd.

* **MINRES (minimum residual method)**: symmetric indefinite $\mathbf{A}$.

* **Bi-CG (bi-conjugate gradient)**: unsymmetric $\mathbf{A}$.

* **Bi-CGSTAB (Bi-CG stabilized)**: improved version of Bi-CG.

* **GMRES (generalized minimum residual method)**: current _de facto_ method for unsymmetric $\mathbf{A}$. E.g., PageRank problem.

* **Lanczos method**: top eigen-pairs of a large symmetric matrix.

* **Arnoldi method**: top eigen-pairs of a large unsymmetric matrix.

* **Lanczos bidiagonalization** algorithm: top singular triplets of large matrix.

* **LSQR**: least square problem $\min \|\mathbf{y} - \mathbf{X} \beta\|_2^2$. Algebraically equivalent to applying CG to the normal equation $(\mathbf{X}^T \mathbf{X} + \lambda^2 I) \beta = \mathbf{X}^T \mathbf{y}$.

* **LSMR**: least square problem $\min \|\mathbf{y} - \mathbf{X} \beta\|_2^2$. Algebraically equivalent to applying MINRES to the normal equation $(\mathbf{X}^T \mathbf{X} + \lambda^2 I) \beta = \mathbf{X}^T \mathbf{y}$.

## Software

### Matlab 

* Iterative methods for solving linear equations:  
    `pcg`, `bicg`, `bicgstab`, `gmres`, ...
* Iterative methods for top eigen-pairs and singular pairs:  
    `eigs`, `svds`, ...
* Pre-conditioner:  
    `cholinc`, `luinc`, ...
    
* Get familiar with the **reverse communication interface (RCI)** for utilizing iterative solvers:
```matlab
x = gmres(A, b)
x = gmres(@Afun, b)
eigs(A)
eigs(@Afun)
```

### Julia

* `eigs` and `svds` in the [Arpack.jl](https://github.com/JuliaLinearAlgebra/Arpack.jl) package. [Numerical examples](http://hua-zhou.github.io/teaching/biostatm280-2019spring/slides/17-eigsvd/eigsvd.html#Lanczos/Arnoldi-iterative-method-for-top-eigen-pairs) later.

* [`IterativeSolvers.jl`](https://github.com/JuliaMath/IterativeSolvers.jl) package. [CG numerical examples](http://hua-zhou.github.io/teaching/biostatm280-2019spring/slides/15-iterative/iterative.html#Numerical-examples)

* See the [list](https://jutho.github.io/KrylovKit.jl/stable/#Package-features-and-alternatives-1) of Julia packages for iterative methods.

#### Least squares example

We first generate a test problem. 

In [34]:
Random.seed!(257) # seed
n, p = 10000, 5000
X = sprandn(n, p, 0.005) # iid standard normals with sparsity 0.005
β = ones(p)
y = X * β + randn(n);

##### Dense QR decomposition (LAPACK)

In [35]:
β̂  =  Matrix(X) \ y
# least squares by sparse QR
@benchmark $(Matrix(X)) \ $y

BenchmarkTools.Trial: 1 sample with 1 evaluation per sample.
 Single result which took 10.181 s (0.12% GC) to evaluate,
 with a memory estimate of 383.14 MiB, over 28 allocations.

##### Sparse QR decomposition

In [36]:
β̂_qr = X \ y
@show norm(β̂ - β̂_qr)
# least squares by sparse QR
@benchmark $X \ $y

norm(β̂ - β̂_qr) = 6.467878563856564e-13


BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  2.295 s …   2.425 s  ┊ GC (min … max): 3.50% … 3.85%
 Time  (median):     2.425 s              ┊ GC (median):    3.78%
 Time  (mean ± σ):   2.382 s ± 74.839 ms  ┊ GC (mean ± σ):  3.71% ± 0.19%

  ▁                                                       █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.3 s          Histogram: frequency by time        2.42 s <

 Memory estimate: 2.28 GiB, allocs estimate: 195.

##### IterativeSolvers.jl

LSQR algorithm as implemented in the `IterativeSolvers.jl` package.

In [37]:
β̂_lsqr = IterativeSolvers.lsqr(X, y)
@show norm(β̂_qr - β̂_lsqr)
# least squares by lsqr
@benchmark IterativeSolvers.lsqr($X, $y)

norm(β̂_qr - β̂_lsqr) = 4.659591525976124e-6


BenchmarkTools.Trial: 289 samples with 1 evaluation per sample.
 Range (min … max):  16.610 ms …  29.555 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     17.174 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   17.334 ms ± 925.490 μs  ┊ GC (mean ± σ):  1.04% ± 1.57%

   ▁  ▄█▄ ▂▁ ▂     ▂                                            
  ▇█▄████▇██▇█▇▆█▇▇█▇▆▇▆▆▅▅▅▄▄▃▃▃▃▁▃▄▃▁▃▃▃▁▁▁▃▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▃
  16.6 ms         Histogram: frequency by time         19.5 ms <

 Memory estimate: 5.35 MiB, allocs estimate: 557.

LSMR algorithm as implemented in the `IterativeSolvers.jl` package.

In [38]:
β̂_lsmr = IterativeSolvers.lsmr(X, y)
@show norm(β̂_qr - β̂_lsmr)
# least squares by lsmr
@benchmark IterativeSolvers.lsmr($X, $y)

norm(β̂_qr - β̂_lsmr) = 0.0006005238631124137


BenchmarkTools.Trial: 386 samples with 1 evaluation per sample.
 Range (min … max):  12.281 ms …  25.805 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     12.808 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   12.966 ms ± 813.292 μs  ┊ GC (mean ± σ):  0.60% ± 1.57%

      ▃ ▁▁█ ▆ ▃▂ ▁                                              
  ▄▃▅▇█▇██████████▇▇▄▆▆▅▇▅██▆█▆▆▆▃▆▃▅▄▅▅▄▃▄▄▃▄▄▁▁▃▃▃▃▃▁▁▃▃▁▁▃▃ ▄
  12.3 ms         Histogram: frequency by time         14.3 ms <

 Memory estimate: 2.18 MiB, allocs estimate: 262.

I don't know how to precondition the LSQR/LSMR algorithm in `IterativeSolvers.jl`.

##### Krylov.jl on CPU

LSQR algorithm as implemented in the `Krylov.jl` package.

In [39]:
# without preconditioner
β̂_lsqr = Krylov.lsqr(X, y)
@show norm(β̂_qr - β̂_lsqr[1])
# least squares by lsqr!
workspace = LsqrWorkspace(X, y)
@benchmark Krylov.lsqr!($workspace, $X, $y)

norm(β̂_qr - β̂_lsqr[1]) = 4.659591525876393e-6


BenchmarkTools.Trial: 306 samples with 1 evaluation per sample.
 Range (min … max):  15.815 ms …  27.706 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     16.197 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   16.339 ms ± 751.808 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

          ▃█▁▁▃▂▂ ▂                                             
  ▇▇▄▄▄▆█████████▇█▆▇▅▄▄▆▃▅▄▅▅▄▄▁▃▃▅▄▃▃▅▃▄▄▁▃▃▃▃▃▃▃▃▁▁▃▁▃▁▃▁▃▃ ▄
  15.8 ms         Histogram: frequency by time         17.5 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [40]:
# with preconditioner
pl = Diagonal(vec(sum(abs2, X, dims=1)))
workspace = LsqrWorkspace(X, y)
β̂_lsqr = Krylov.lsqr!(workspace, X, y, N = pl, ldiv = true)
@show norm(β̂_qr - β̂_lsqr.x)
@benchmark Krylov.lsqr!($workspace, $X, $y, N = $pl, ldiv = true)

norm(β̂_qr - β̂_lsqr.x) = 4.04134803933291e-6


BenchmarkTools.Trial: 337 samples with 1 evaluation per sample.
 Range (min … max):  14.550 ms …  15.852 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     14.815 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   14.878 ms ± 240.527 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

       ▂▂▂▄▇█▄▆█   ▂▂▄                                          
  ▆▃▆▆▆█████████▆▆████▇▄▃▅▄▅▄▃▅▁▃▄▃▃▄▃▃▃▃▃▁▃▁▃▁▁▁▁▃▁▃▃▁▃▃▁▃▄▁▃ ▄
  14.6 ms         Histogram: frequency by time         15.8 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

LSMR algorithm as implemented in the `Krylov.jl` package.

In [41]:
# without preconditioner
workspace = LsmrWorkspace(X, y)
β̂_lsmr = Krylov.lsmr!(workspace, X, y)
@show β̂_lsmr
@show norm(β̂_qr - β̂_lsmr.x)
# least squares by lsmr
@benchmark Krylov.lsmr!($workspace, $X, $y)

β̂_lsmr = ┌──────────────────┬──────────────────┬────────────────────┐
│     LsmrWorkspace│      nrows: 10000│         ncols: 5000│
├──────────────────┼──────────────────┼────────────────────┤
│Precision: Float64│ Architecture: CPU│Storage: 351.729 KiB│
├──────────────────┼──────────────────┼────────────────────┤
│         Attribute│              Type│                Size│
├──────────────────┼──────────────────┼────────────────────┤
│                 m│             Int64│             8 bytes│
│                 n│             Int64│             8 bytes│
│                 x│   Vector{Float64}│          39.062 KiB│
│                Nv│   Vector{Float64}│          39.062 KiB│
│               Aᴴu│   Vector{Float64}│          39.062 KiB│
│                 h│   Vector{Float64}│          39.062 KiB│
│              hbar│   Vector{Float64}│          39.062 KiB│
│                Mu│   Vector{Float64}│          78.125 KiB│
│                Av│   Vector{Float64}│          78.125 KiB│
│           er

BenchmarkTools.Trial: 306 samples with 1 evaluation per sample.
 Range (min … max):  15.826 ms …  24.918 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     16.235 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   16.387 ms ± 646.723 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

         ▆█▅▆▅ ▃▂▃ ▃   ▃▂                                       
  ▅▄▇▅▆▇██████████▆█▇▅▅██▆▇▄▇▇▃▄▃▆▄▆▁▅▆▁▄▆▄▆▃▃▄▅▃▅▁▄▁▄▁▇▃▄▃▃▁▃ ▄
  15.8 ms         Histogram: frequency by time         17.4 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

In [42]:
# with preconditioner
pl = Diagonal(vec(sum(abs2, X, dims=1)))
workspace = LsmrWorkspace(X, y)
β̂_lsmr = Krylov.lsmr!(workspace, X, y, N = pl, ldiv = true)
@show β̂_lsmr
@show norm(β̂_qr - β̂_lsmr.x)

β̂_lsmr = ┌──────────────────┬──────────────────┬────────────────────┐
│     LsmrWorkspace│      nrows: 10000│         ncols: 5000│
├──────────────────┼──────────────────┼────────────────────┤
│Precision: Float64│ Architecture: CPU│Storage: 390.791 KiB│
├──────────────────┼──────────────────┼────────────────────┤
│         Attribute│              Type│                Size│
├──────────────────┼──────────────────┼────────────────────┤
│                 m│             Int64│             8 bytes│
│                 n│             Int64│             8 bytes│
│                 x│   Vector{Float64}│          39.062 KiB│
│                Nv│   Vector{Float64}│          39.062 KiB│
│               Aᴴu│   Vector{Float64}│          39.062 KiB│
│                 h│   Vector{Float64}│          39.062 KiB│
│              hbar│   Vector{Float64}│          39.062 KiB│
│                Mu│   Vector{Float64}│          78.125 KiB│
│                Av│   Vector{Float64}│          78.125 KiB│
│             

8.476970605853465e-6

In [43]:
# with preconditioner
pl = Diagonal(vec(sum(abs2, X, dims=1)))
workspace = LsmrWorkspace(X, y)
β̂_lsmr = Krylov.lsmr!(workspace, X, y, N = pl, ldiv = true)
@show β̂_lsmr
@show norm(β̂_qr - β̂_lsmr.x)
@benchmark Krylov.lsmr!($workspace, $X, $y, N = $pl, ldiv = true)

β̂_lsmr = ┌──────────────────┬──────────────────┬────────────────────┐
│     LsmrWorkspace│      nrows: 10000│         ncols: 5000│
├──────────────────┼──────────────────┼────────────────────┤
│Precision: Float64│ Architecture: CPU│Storage: 390.791 KiB│
├──────────────────┼──────────────────┼────────────────────┤
│         Attribute│              Type│                Size│
├──────────────────┼──────────────────┼────────────────────┤
│                 m│             Int64│             8 bytes│
│                 n│             Int64│             8 bytes│
│                 x│   Vector{Float64}│          39.062 KiB│
│                Nv│   Vector{Float64}│          39.062 KiB│
│               Aᴴu│   Vector{Float64}│          39.062 KiB│
│                 h│   Vector{Float64}│          39.062 KiB│
│              hbar│   Vector{Float64}│          39.062 KiB│
│                Mu│   Vector{Float64}│          78.125 KiB│
│                Av│   Vector{Float64}│          78.125 KiB│
│             

BenchmarkTools.Trial: 330 samples with 1 evaluation per sample.
 Range (min … max):  14.539 ms …  30.140 ms  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     15.024 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   15.192 ms ± 943.843 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

   ▂    █▆▃ ▄▃ ▁   ▁                                            
  ▇█▆█▄█████████▆█▆█▇▇█▅▅▆▆▅▆▇▅▄▇▄▃▃▃▃▄▃▃▁▃▃▃▃▃▁▃▁▃▃▃▁▁▃▁▁▁▁▁▃ ▄
  14.5 ms         Histogram: frequency by time         16.8 ms <

 Memory estimate: 0 bytes, allocs estimate: 0.

##### Krylov.jl on GPU

Krylov.jl functions run on [GPUs](https://jso.dev/Krylov.jl/stable/). See my other notebook for details.

#### Use LinearMaps in iterative solvers

In many applications, it is advantageous to define linear maps indead of forming the actual (sparse) matrix. For a linear map, we need to specify how it acts on right- and left-multiplication on a vector. The [`LinearMaps.jl`](https://github.com/Jutho/LinearMaps.jl) package is exactly for this purpose and interfaces nicely with `IterativeSolvers.jl`, `Arnoldi.jl` and other iterative solver packages.

Applications:  
1. The matrix is not sparse but admits special structure, e.g., easy + low rank (PageRank), Kronecker proudcts, etc.  
2. Less memory usage. 
3. Linear algebra on a standardized (centered and scaled) sparse matrix.

Consider the differencing operator that takes differences between neighboring pixels

$$
\mathbf{D} = \begin{pmatrix}
-1 & 1 & & & \\
& -1 & 1 & & \\
& & \ddots & \\
& & & - 1 & 1 \\
1 & & & & -1
\end{pmatrix}.
$$

In [44]:
using LinearMaps, IterativeSolvers

# Overwrite y with A * x
# left difference assuming periodic boundary conditions
function leftdiff!(y::AbstractVector, x::AbstractVector) 
    N = length(x)
    length(y) == N || throw(DimensionMismatch())
    @inbounds for i in 1:N
        y[i] = x[i] - x[mod1(i - 1, N)]
    end
    return y
end

# Overwrite y with A' * x
# minus right difference
function mrightdiff!(y::AbstractVector, x::AbstractVector) 
    N = length(x)
    length(y) == N || throw(DimensionMismatch())
    @inbounds for i in 1:N
        y[i] = x[i] - x[mod1(i + 1, N)]
    end
    return y
end

# define linear map
D = LinearMap{Float64}(leftdiff!, mrightdiff!, 100; ismutating=true) 

100×100 FunctionMap{Float64,true}(leftdiff!, mrightdiff!; issymmetric=false, ishermitian=false, isposdef=false)

Linear maps can be used like a regular matrix.

In [45]:
@show size(D)
v = ones(size(D, 2)) # vector of all 1s
@show D * v
@show D' * v;

size(D) = (100, 100)
D * v = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
D' * v = [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 

If we form the corresponding dense matrix, it will look like

In [46]:
Matrix(D)

100×100 Matrix{Float64}:
  1.0   0.0   0.0   0.0   0.0   0.0  …   0.0   0.0   0.0   0.0   0.0  -1.0
 -1.0   1.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0  -1.0   1.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0  -1.0   1.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0  -1.0   1.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0  -1.0   1.0  …   0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0  -1.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  ⋮                             ⋮    ⋱         ⋮                      
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.0   0.0   0.0   0.0   0.0   0.0      0.0   0.0   0.0   0.0   0.0   0.0
  0.

If we form the corresponding sparse matrix, it will look like

In [47]:
using SparseArrays
sparse(D)

100×100 SparseMatrixCSC{Float64, Int64} with 200 stored entries:
⎡⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⎤
⎢⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⎦

In [48]:
using UnicodePlots
spy(sparse(D))

       ┌──────────────────────────────┐    
     1 │⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁│ > 0
       │⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│ < 0
       │⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀⠀⠀│    
       │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⣄⠀⠀│    
   100 │⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠳⠄│    
       └──────────────────────────────┘    
       ⠀1⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀100⠀    
       ⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀200 ≠ 0⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀    

Compute top singular values using iterative method (Arnoldi).

In [49]:
using Arpack
Arpack.svds(D, nsv = 3)

(SVD{Float64, Float64, Matrix{Float64}, Vector{Float64}}([0.09999999999994835 0.12610107456829475 -0.06401967660589121; -0.09999999999994832 -0.12987207165686598 0.0559753964126666; … ; 0.09999999999995117 0.11708293685000323 -0.07931951776567879; -0.09999999999994938 -0.1218324141485034 0.07181130038336052], [2.000000000000001, 1.9990131207314643, 1.9990131207314612], [0.099999999999948 -0.09999999999994859 … 0.09999999999995006 -0.09999999999994866; 0.1280497579383058 -0.13156621732034646 … 0.11951664975119568 -0.12402794466205232; -0.06002715628732366 0.051868395579741794 … -0.07560271445028788 0.0679490172328382]), 3, 32, 523, [0.22689451355330165, 0.06017972265709556, 0.02109087446432143, -0.08110676369020713, -0.17325843516179368, -0.019567304450968628, -0.20925235785590932, -0.07653571129483476, 0.09479144705912801, 0.045410977125435986  …  -0.11226794256146168, -0.28850472721154824, 0.08500546814634621, 0.008513756735830841, -0.06919645518583611, 0.08081444805081334, 0.09503283

In [50]:
using LinearAlgebra
# check solution against the direct method for SVD
svdvals(Matrix(D))

100-element Vector{Float64}:
 2.0
 1.9990131207314632
 1.999013120731463
 1.9960534568565436
 1.996053456856543
 1.9911239292061602
 1.99112392920616
 1.9842294026289558
 1.9842294026289553
 1.9753766811902755
 ⋮
 0.2506664671286085
 0.2506664671286083
 0.18821662663702862
 0.18821662663702837
 0.12558103905862675
 0.12558103905862664
 0.06282151815625657
 0.06282151815625645
 3.355901641602888e-16

Compute top eigenvalues of the Gram matrix `D'D` using iterative method (Arnoldi).

In [51]:
Arpack.eigs(D'D, nev = 3, which = :LM)

([4.000000000000006, 3.9960534568565613, 3.9960534568565387], [-0.10000000000000824 0.009399690541647463 0.14110863126584344; 0.1000000000000051 -0.0005208581322662039 -0.14142039706776885; … ; -0.10000000000001495 0.02701117221455527 0.13881785395111335; 0.10000000000001161 -0.018241426666770824 -0.1402399741627136], 3, 32, 524, [0.08156332551188875, 0.06815388206864158, -0.03395372999522577, 0.07692963746501648, 0.07537749074645597, 0.015946689072540807, 0.028370443731959637, 0.023607902842913576, 0.0014420366606339664, -0.1276563953611981  …  0.06304883341782141, -0.08555036993075614, 0.029425027599579375, 0.2564476034917131, 0.22307026896595492, 0.06199052229404638, 0.09402616761028919, 0.08407304386123257, -0.0634808930891989, -0.011886292077481962])

## Further reading

* Chapter 5 of [Numerical Optimization](https://ucla.worldcat.org/title/numerical-optimization/oclc/209918411&referer=brief_results) by Jorge Nocedal and Stephen Wright (1999).

* Sections 11.3-11.5 of [Matrix Computations](https://ucla.worldcat.org/title/matrix-computations/oclc/824733531&referer=brief_results) by Gene Golub and Charles Van Loan (2013).